# Phonon Density of States + Dispersions

Compute the phonon **dispersion curves** and phonon **density of states (DOS)** of a single material using Quantum ESPRESSO on the Mat3ra platform.

The workflow runs, in sequence: SCF (`pw.x`) → DFPT on a q-grid (`ph.x`) → interatomic force constants (`q2r.x`) → phonon DOS on a q-grid and dispersion along a high-symmetry path (`matdyn.x`).

Single material; no reference materials or prerequisites. Phonon DFPT cost scales steeply with the number of atoms, so use a **small primitive cell**.

<h2 style="color:green">Usage</h2>

1. Set the material and parameters in cells 1.2 / 1.3 below (default: **Silicon**, which gives clean phonons out of the box). For harder cases (e.g. Graphene / 2D) use the controls in 1.3 — relaxation, denser k/q grids, higher cutoffs — to avoid imaginary modes.
2. Click "Run" > "Run All".
3. Inspect the phonon dispersion curves and DOS at the end. Acoustic branches should go to 0 at Γ, with no large imaginary (negative) modes.

## Summary

1. Set up the environment and parameters.
2. Authenticate and initialize API client.
3. Load the material.
4. Configure the Phonon DOS + Dispersions workflow.
5. Configure compute.
6. Create the job.
7. Submit and monitor the job.
8. Retrieve and visualize the phonon dispersion and DOS.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"  # small primitive cell (FCC, mp-149); phonon DFPT is expensive

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "phonon_dos_dispersion.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Phonon Density of States + Dispersions"

# Model parameters
MODEL_SUBTYPE = "gga"  # "gga" or "lda"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set specific phonon parameters

In [ ]:
# Method parameters
PSEUDOPOTENTIAL_TYPE = "us"  # "us" (ultrasoft), "nc" (norm-conserving), "paw"
FUNCTIONAL = "pbe"           # for gga: "pbe", "pbesol"; for lda: "pz"

# Relax the structure before phonons. Phonons require a force-minimized geometry —
# an unrelaxed structure produces imaginary (negative) modes. Recommended for any
# material not already at its equilibrium geometry (e.g. Graphene / 2D).
ADD_RELAXATION = False

# Electronic k-grids; if None, the workflow default is used
SCF_KGRID = None         # e.g. [8, 8, 8] (3D) or [12, 12, 1] (2D)
RELAXATION_KGRID = None  # e.g. [8, 8, 8]  (used only when ADD_RELAXATION)

# Phonon q-grid for the DFPT run (ph.x); if None, the workflow default is used
PHONON_QGRID = None      # e.g. [4, 4, 4] (3D) or [6, 6, 1] (2D)

# Phonon DOS interpolation grid (matdyn.x); if None, the workflow default is used
DOS_QGRID = None         # e.g. [12, 12, 12]

# Phonon dispersion path (matdyn.x); if None, the workflow default is used
QPATH = None             # e.g. [{"point": "G", "steps": 20}, {"point": "M", "steps": 20},
                         #       {"point": "K", "steps": 20}, {"point": "G", "steps": 20}]

# Energy cutoffs (raise for harder cases — e.g. 60 / 480 for Graphene)
ECUTWFC = 40
ECUTRHO = 200

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

material.basis.set_labels_from_list([])
saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

## 4. Configure workflow
### 4.1. Select application

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load workflow from Standata and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Add relaxation (optional)
Prepend a relaxation subworkflow so the phonons are computed on a force-minimized structure.

In [ ]:
if ADD_RELAXATION:
    workflow.add_relaxation()
    visualize_workflow(workflow)

### 4.4. Set model and its parameters (physics)

In [ ]:
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.mode import ModelFactory

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

for subworkflow in workflow.subworkflows:
    subworkflow.model = model

visualize_workflow(workflow)

### 4.5. Modify method (computational parameters)
Optional overrides — each applies only if its parameter is set above. Electronic k-grid + cutoffs on `pw.x`; phonon q-grid on `ph.x`; DOS interpolation grid and dispersion path on `matdyn.x`.

In [ ]:
from mat3ra.wode.context.providers import (
    PlanewaveCutoffsContextProvider, PointsGridDataProvider, PointsPathDataProvider,
)

# The phonon subworkflow is index 0, or 1 when a relaxation subworkflow is prepended.
phonon_subworkflow = workflow.subworkflows[1 if ADD_RELAXATION else 0]


def set_grid(subworkflow, unit_name, dimensions, name=None):
    unit = subworkflow.get_unit_by_name(name=unit_name)
    if unit:
        unit.add_context(PointsGridDataProvider(dimensions=dimensions, isEdited=True, **({"name": name} if name else {})).get_context_item_data())
        subworkflow.set_unit(unit)


def set_path(subworkflow, unit_name, path, name):
    unit = subworkflow.get_unit_by_name(name=unit_name)
    if unit:
        unit.add_context(PointsPathDataProvider(path=path, isEdited=True, name=name).get_context_item_data())
        subworkflow.set_unit(unit)


# Relaxation electronic k-grid
if RELAXATION_KGRID is not None and ADD_RELAXATION:
    unit = workflow.subworkflows[0].get_unit_by_name(name_regex="relax")
    if unit:
        unit.add_context(PointsGridDataProvider(dimensions=RELAXATION_KGRID, isEdited=True).get_context_item_data())
        workflow.subworkflows[0].set_unit(unit)

# SCF electronic k-grid (pw.x)
if SCF_KGRID is not None:
    set_grid(phonon_subworkflow, "pw_scf", SCF_KGRID)

# Phonon q-grid for DFPT (ph.x)
if PHONON_QGRID is not None:
    set_grid(phonon_subworkflow, "ph_grid", PHONON_QGRID, name="qgrid")

# Phonon DOS interpolation grid (matdyn.x)
if DOS_QGRID is not None:
    set_grid(phonon_subworkflow, "matdyn_grid", DOS_QGRID, name="igrid")

# Phonon dispersion path (matdyn.x)
if QPATH is not None:
    set_path(phonon_subworkflow, "matdyn_path", QPATH, name="ipath")

# Energy cutoffs on every pw.x unit
if ECUTWFC is not None:
    cutoffs_context = PlanewaveCutoffsContextProvider(
        wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True
    ).get_context_item_data()
    for unit_name in ["pw_relax", "pw_vc-relax", "pw_scf"]:
        for swf in workflow.subworkflows:
            unit = swf.get_unit_by_name(name=unit_name)
            if unit:
                unit.add_context(cutoffs_context)
                swf.set_unit(unit)

### 4.6. Preview final workflow

In [ ]:
visualize_workflow(workflow)

### 4.7. Save workflow to collection

In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.ui import display_JSON

job_name = f"{MY_WORKFLOW_NAME} {saved_material.formula} {timestamp}"
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print(f"✅ Job created: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve and visualize results
### 8.1. Phonon dispersion

In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

phonon_dispersions_data = get_properties_for_job(
    client, job_id, property_name=PropertyName.non_scalar.phonon_dispersions.value
)
visualize_properties(phonon_dispersions_data, title="Phonon Dispersion", extra_config={"material": material.to_dict()})

### 8.2. Phonon density of states

In [ ]:
phonon_dos_data = get_properties_for_job(
    client, job_id, property_name=PropertyName.non_scalar.phonon_dos.value
)
visualize_properties(phonon_dos_data, title="Phonon DOS", extra_config={"material": material.to_dict()})